<a href="https://colab.research.google.com/github/SilkSherstka/hse_python_lessons/blob/main/Andronova_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 0. Запустите эту ячейку, чтобы скачать файл для работы

In [4]:
!wget https://raw.githubusercontent.com/vifirsanova/hse-python-course/refs/heads/main/data.txt

--2025-12-22 13:42:58--  https://raw.githubusercontent.com/vifirsanova/hse-python-course/refs/heads/main/data.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2382 (2.3K) [text/plain]
Saving to: ‘data.txt.1’

data.txt.1          100%[===================>]   2.33K  --.-KB/s    in 0s      

2025-12-22 13:42:58 (16.7 MB/s) - ‘data.txt.1’ saved [2382/2382]



### Часть 1: чтение файла

Напишите функцию read_chat_log(file_path), которая читает файл data.txt и возвращает список строк

In [13]:
def read_chat_log(file_path):
  with open(file_path, 'r') as f:
    return [line.strip() for line in f]

### Часть 2: парсинг данных (создание словаря)

Напишите функцию parse_log_entry(entry), которая разбирает каждую строку файла строку и возвращает список словарей с такими ключами:

- date,
- time,
- level,
- username,
- message

Т.е. для каждой строки создаётся словарь следующего вида:

```
{
    'date': '2024-12-20',
    'time': '10:30:15',
    'level': 'INFO',
    'username': 'Алексей',
    'message': 'Привет всем! Как дела?'
}
```

Обратите внимания, что результатом работы функции должен быть **список словарей**

> Используйте регулярные выражения или слайсинг (на выбор!) для разбиения данных по нужным ключам

In [14]:
def parse_log_entry(entry):
    result = []

    for line in entry:
        log_dict = {}

        first_bracket_end = line.find(']')
        second_bracket_start = line.find('[', first_bracket_end + 1)
        second_bracket_end = line.find(']', second_bracket_start + 1)
        colon_pos = line.find(':', second_bracket_end + 1)

        datetime_str = line[1:first_bracket_end]  # дата и время
        date, time = datetime_str.split()

        # уровень логгирования
        level = line[second_bracket_start + 1:second_bracket_end]

        # имя пользователя
        username = line[second_bracket_end + 2:colon_pos]

        # сообщение
        message = line[colon_pos + 2:]  # +2 чтобы пропустить ": "

        log_dict['date'] = date
        log_dict['time'] = time
        log_dict['level'] = level
        log_dict['username'] = username
        log_dict['message'] = message

        result.append(log_dict)

    return result

parsed_logs = parse_log_entry(parsed_msgs)
print(parsed_logs)

[{'date': '2024-12-16', 'time': '09:15:00', 'level': 'INFO', 'username': 'Алексей', 'message': 'Доброе утро всем! Начинаем рабочий день'}, {'date': '2024-12-16', 'time': '09:20:30', 'level': 'VARIA', 'username': 'Мария', 'message': 'Утро доброе! Кофе кто пил?'}, {'date': '2024-12-16', 'time': '09:21:15', 'level': 'INFO', 'username': 'Иван', 'message': 'Я уже второй)'}, {'date': '2024-12-16', 'time': '09:25:00', 'level': 'INFO', 'username': 'Алексей', 'message': 'План на сегодня: 1) Завершить проект Х 2) Провести созвон с клиентом'}, {'date': '2024-12-16', 'time': '10:30:00', 'level': 'WARN', 'username': 'Система', 'message': 'Нагрузка на сервер 85%'}, {'date': '24-12-16', 'time': '11:45:30', 'level': 'INFO', 'username': 'Мария', 'message': 'Созвон с клиентом переносится на 14:00'}, {'date': '2024-12-16', 'time': '12:00:00', 'level': 'INFO', 'username': 'Алексей', 'message': 'Ок, обновлю календарь'}, {'date': '2024-12-16', 'time': '13:15:22', 'level': 'ERROR', 'username': 'Сервер', 'mes

### Часть 3: валидация данных

Напишите функцию validate_parsed_logs(parsed_logs), которая:

1. Проверяет, что дата имеет формат ГГГГ-ММ-ДД (проверяем только по длине, не по содержимому, т.е. используем условия и проверку по len)

2. Проверяет, что время имеет формат ЧЧ:ММ:СС (также проверяем только по длине)

3. Проверяет, что уровень важности является одним из допустимых: INFO, WARN, ERROR, CRITICAL, DEBUG (сохраните эти уровни во множество)

Функция должна выводить результаты через print() в формате:

```
Всего записей: 8
Корректных: 6
Некорректных: 2

Ошибки:
- Неправильная дата: 1
- Неправильное время: 1
- Неправильный уровень: 0

Некорректные записи:
1. Дата '24-12-10': не соответствует формату ГГГГ-ММ-ДД
2. Время '00:6:54': не соответствует формату ЧЧ:ММ:СС
```

In [15]:
def validate_parsed_logs(parsed_logs):
    valid_levels = {'INFO', 'WARN', 'ERROR', 'CRITICAL', 'DEBUG'}

    total_records = len(parsed_logs)
    correct_records = 0
    incorrect_date_count = 0
    incorrect_time_count = 0
    incorrect_level_count = 0
    incorrect_records = []

    for i, log in enumerate(parsed_logs, 1):
        is_correct = True
        errors = []

        if len(log['date']) != 10:
            is_correct = False
            incorrect_date_count += 1
            errors.append(f"Дата '{log['date']}': не соответствует формату ГГГГ-ММ-ДД")

        if len(log['time']) != 8:
            is_correct = False
            incorrect_time_count += 1
            errors.append(f"Время '{log['time']}': не соответствует формату ЧЧ:ММ:СС")

        if log['level'] not in valid_levels:
            is_correct = False
            incorrect_level_count += 1
            errors.append(f"Уровень '{log['level']}': не является допустимым (допустимые: {', '.join(sorted(valid_levels))})")

        if not is_correct:
            incorrect_records.append((i, errors))
        else:
            correct_records += 1

    print(f"Всего записей: {total_records}")
    print(f"Корректных: {correct_records}")
    print(f"Некорректных: {total_records - correct_records}\n")

    print("Ошибки:")
    print(f"- Неправильная дата: {incorrect_date_count}")
    print(f"- Неправильное время: {incorrect_time_count}")
    print(f"- Неправильный уровень: {incorrect_level_count}\n")

    if incorrect_records:
        print("Некорректные записи:")
        for record_num, errors in incorrect_records:
            print(f"{record_num}. {errors[0]}")
            # Если есть дополнительные ошибки для этой записи
            for error in errors[1:]:
                print(f"   {error}")
    else:
        print("Некорректные записи: нет")

validate_parsed_logs(parsed_logs)

Всего записей: 25
Корректных: 21
Некорректных: 4

Ошибки:
- Неправильная дата: 2
- Неправильное время: 1
- Неправильный уровень: 1

Некорректные записи:
2. Уровень 'VARIA': не является допустимым (допустимые: CRITICAL, DEBUG, ERROR, INFO, WARN)
6. Дата '24-12-16': не соответствует формату ГГГГ-ММ-ДД
10. Дата '24-12-16': не соответствует формату ГГГГ-ММ-ДД
20. Время '3:45:00': не соответствует формату ЧЧ:ММ:СС


### Теоретический вопрос

Что такое режимы открытия файлов ('r', 'w', 'a', 'rb', 'wb')? Как читать и записывать данные в файл? Что такое контекстный менеджер with и зачем он нужен?

'r' - только чтение. Файл должен существовать

'w' - запись. Создаёт файл или полностью перезаписывает существующий

'a' - добавление в конец файла. Старое содержимое сохраняется

'rb' - чтение в бинарном режиме (для картинок, видео и т.д.)

'wb' - запись в бинарном режиме

In [12]:
# чтение
file = open('файл.txt', 'r', encoding='utf-8')
text = file.read()
file.close()

# запись
file = open('файл.txt', 'w', encoding='utf-8')
file.write('Первая строка\n')
file.write('Вторая строка')
file.close()

Контекстный менеджер with - это обертка, которая сама открывает и закрывает файл (даже если ошибка была во время работы). Нужен для автоматического закрытия файла. Без него можно забыть закрыть файл, и это вызовет проблемы.